In [15]:
import torch
from torch import nn

# naive self-attention

In [2]:
inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your
    [0.55, 0.87, 0.66], # journey
    [0.57, 0.85, 0.64], # starts
    [0.22, 0.58, 0.33], # with
    [0.77, 0.25, 0.10], # one
    [0.05, 0.80, 0.55]  # step
])

query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [3]:
attn_scores_2_true = inputs @ query
print(attn_scores_2_true)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [4]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print(attn_weights_2_tmp)
print(attn_weights_2_tmp.sum())

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
tensor(1.0000)


In [5]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum()

attn_weights_2_naive = softmax_naive(attn_scores_2)
print(attn_weights_2_naive)
print(attn_weights_2_naive.sum())

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.)


In [6]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print(attn_weights_2)
print(attn_weights_2.sum())

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.)


In [7]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [8]:
context = attn_weights_2 @ inputs # journey context vector
print(context)

tensor([0.4419, 0.6515, 0.5683])


In [9]:
attn_scores_1 = inputs @ inputs[0]
attn_weights_1 = torch.softmax(attn_scores_1, dim=0)
context_1 = attn_weights_1 @ inputs # 'your' context vector
print(context_1)

tensor([0.4421, 0.5931, 0.5790])


In [10]:
attn_scores_2 = inputs @ inputs[1]
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
context_2 = attn_weights_2 @ inputs # 'journey' context vector
print(context_2)

tensor([0.4419, 0.6515, 0.5683])


In [11]:
# all at once
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=1)
context = attn_weights @ inputs
print(context)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


# trainable self-attention

In [12]:
inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your
    [0.55, 0.87, 0.66], # journey
    [0.57, 0.85, 0.64], # starts
    [0.22, 0.58, 0.33], # with
    [0.77, 0.25, 0.10], # one
    [0.05, 0.80, 0.55]  # step
])

x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)
print(key_2)
print(value_2)

keys = inputs @ W_key
values = inputs @ W_value
print(keys.shape)
print(values.shape)

tensor([0.7209, 0.5739])
tensor([1.4724, 1.0752])
tensor([1.6225, 0.5441])
torch.Size([6, 2])
torch.Size([6, 2])


In [13]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

context_2 = attn_weights_2 @ values
print(context_2)

tensor(1.6785)
tensor([1.4666, 1.6785, 1.6691, 0.8477, 1.0316, 1.0030])
tensor([0.1846, 0.2144, 0.2130, 0.1192, 0.1357, 0.1330])
tensor([1.2800, 0.4199])


In [14]:
queries = inputs @ W_query
keys = inputs @ W_key
values = inputs @ W_value

attn_scores = queries @ keys.T
d_k = keys.shape[-1]
attn_weights = torch.softmax(queries @ keys.T / d_k**0.5, dim=-1)

context = attn_weights @ values

print(queries.shape)       # (6, 2)
print(keys.shape)          # (6, 2)
print(values.shape)        # (6, 2)

print(attn_scores.shape)   # (6, 6)
print(attn_weights.shape)  # (6, 6)
print(context.shape)       # (6, 2)

assert torch.allclose(context[1], context_2)

torch.Size([6, 2])
torch.Size([6, 2])
torch.Size([6, 2])
torch.Size([6, 6])
torch.Size([6, 6])
torch.Size([6, 2])


In [16]:
class SelfAttention_V1(torch.nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = torch.nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = torch.nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = torch.nn.Parameter(torch.rand(d_in, d_out))
        
    def forward(self, x):
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value
        
        attn_scores = queries @ keys.T
        d_k = keys.shape[-1]
        
        context = torch.softmax(attn_scores / d_k**0.5, dim=-1) @ values
        return context
        

In [26]:
torch.manual_seed(123)
sa_v1 = SelfAttention_V1(d_in, d_out)
sa_v1(inputs)


tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [24]:
class SelfAttention_V2(torch.nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
    def forward(self, x):
        queries = self.W_query(x) 
        keys = self.W_key(x) 
        values = self.W_value(x) 
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context = attn_weights @ values
        return context
        

In [27]:
torch.manual_seed(789)
sa_v2 = SelfAttention_V2(d_in, d_out)
sa_v2(inputs)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)